# 02 — LightGBM Feature Ablation Study (Zero-Fee) v0

**Task ID:** `MODEL-LGBM-EXPLORATION-003`  
**SDD:** `docs/design/LGBM_Modeling_Strategy_SDD.md`  
**Input:** `BTCUSDT_5m_structural.parquet` + `feature_registry_v2.json`  
**Note:** This notebook uses the **original (pre-patch)** feature parquet. MTF features
carry lookahead bias — see `v1` for the corrected run.

## Objective

Conduct a controlled **feature ablation study** to determine which thematic feature
group carries the most predictive signal for BTC/USDT 5-minute price moves.
All evaluation is done at **zero fees** to isolate pure signal quality.

## The Four Ablation Models

| Model | Registry Key | n Features | Hypothesis |
|-------|-------------|-----------|------------|
| M1 — Structure | `structure_only` | 11 | Swing geometry alone predicts breakouts |
| M2 — Struct + Liquidity | `market_structure` | 21 | Liquidity context amplifies swing signals |
| M3 — Volatility | `volatility_only` | 8 | Squeeze-release dynamics drive moves |
| M4 — Omni | `all` | 39 | Full feature set upper-bound reference |

In [ ]:
import json
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterGrid

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)

# ── TBM parameters ──────────────────────────────────────────────────────────
TBM_TP_MULT  = 2.5   # Take-profit: 2.5 × ATR_72 (6h rolling ATR on 5m bars)
TBM_SL_MULT  = 1.5   # Stop-loss:   1.5 × ATR_72 → asymmetric 1.67 gross RR
TBM_HORIZON  = 288   # Time barrier: 288 bars × 5m = 24 hours

# ── Signal threshold ────────────────────────────────────────────────────────
PROB_THRESHOLD     = 0.75
PROB_THRESHOLD_ALT = 0.65

# ── Chronological split dates ────────────────────────────────────────────────
TRAIN_END = "2023-12-31 23:55"
VAL_END   = "2024-12-31 23:55"

# ── LightGBM base parameters ────────────────────────────────────────────────
BASE_LGB_PARAMS = dict(
    objective         = "binary",
    metric            = "auc",
    verbose           = -1,
    n_jobs            = -1,
    seed              = 42,
    n_estimators      = 1000,
    min_child_samples = 50,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
)

# ── Constrained hyperparameter grid (32 configs) ────────────────────────────
PARAM_GRID = list(ParameterGrid({
    "num_leaves":       [15, 31],
    "max_depth":        [4, 6],
    "learning_rate":    [0.01, 0.05],
    "colsample_bytree": [0.6, 0.8],
    "subsample":        [0.6, 0.8],
}))

# ── Fee survival gate ─────────────────────────────────────────────────────────
FEE_ROUNDTRIP   = 0.0010
EV_SURVIVAL_MIN = 0.0040

print(f"Grid size: {len(PARAM_GRID)} configs × 4 models = {len(PARAM_GRID)*4} total fits")
print(f"TBM: TP={TBM_TP_MULT}×ATR  SL={TBM_SL_MULT}×ATR  Horizon={TBM_HORIZON} bars ({TBM_HORIZON*5/60:.0f}h)")

## Phase A — Load & Merge Data

In [ ]:
def find_repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise RuntimeError("pyproject.toml not found")

REPO_ROOT    = find_repo_root()
RAW_DIR      = REPO_ROOT / "data" / "raw"
FEATURES_DIR = REPO_ROOT / "data" / "features"
CACHE_DIR    = REPO_ROOT / "data" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OHLCV_PATH    = RAW_DIR      / "BTCUSDT_5m.parquet"
FEATURES_PATH = FEATURES_DIR / "BTCUSDT_5m_structural.parquet"
REGISTRY_PATH = FEATURES_DIR / "feature_registry_v2.json"
LABELS_CACHE  = CACHE_DIR    / "BTCUSDT_5m_tbm_labels.parquet"

print("Loading raw OHLCV ...")
df_raw = pd.read_parquet(OHLCV_PATH, columns=["open", "high", "low", "close", "volume"])
print(f"  Raw:  {len(df_raw):,} bars  {df_raw.index[0].date()} → {df_raw.index[-1].date()}")

print("Loading structural features ...")
df_feat = pd.read_parquet(FEATURES_PATH)
print(f"  Feat: {len(df_feat):,} bars  {df_feat.shape[1]} columns")

df = df_raw.join(df_feat, how="inner").dropna()
print(f"\nMerged: {len(df):,} bars  {df.shape[1]} columns")
print(f"Date range: {df.index[0]} → {df.index[-1]}")
assert df.isnull().sum().sum() == 0

## Phase A — Feature Subset Definitions from Registry

In [ ]:
with open(REGISTRY_PATH) as f:
    registry = json.load(f)

print(f"Registry version: {registry['schema_version']}  |  Total features: {registry['total_features']}")

ABLATION_MODELS = {
    "M1_Structure":        registry["ablation_subsets"]["structure_only"],
    "M2_Struct_Liquidity": registry["ablation_subsets"]["market_structure"],
    "M3_Volatility":       registry["ablation_subsets"]["volatility_only"],
    "M4_Omni":             registry["ablation_subsets"]["all"],
}

print("\nFeature availability check:")
for model_name, feats in ABLATION_MODELS.items():
    missing = [f for f in feats if f not in df.columns]
    status  = "OK" if not missing else f"MISSING: {missing}"
    print(f"  {model_name:<25}  {len(feats):>2} features  → {status}")

## Phase A — Chronological Split

In [ ]:
train_mask = df.index <= TRAIN_END
val_mask   = (df.index > TRAIN_END) & (df.index <= VAL_END)
test_mask  = df.index > VAL_END

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()   # held out — not evaluated here

n_val_days   = (df_val.index[-1]  - df_val.index[0]).days + 1
n_train_days = (df_train.index[-1] - df_train.index[0]).days + 1

print(f"Train:      {len(df_train):>7,} bars  {df_train.index[0].date()} → {df_train.index[-1].date()}")
print(f"Validation: {len(df_val):>7,} bars  {df_val.index[0].date()} → {df_val.index[-1].date()}  ({n_val_days} days)")
print(f"Test:       {len(df_test):>7,} bars  {df_test.index[0].date()} → {df_test.index[-1].date()}  [HELD OUT]")

## Phase B — Triple Barrier Method Labeling

In [ ]:
def compute_tbm_labels(
    close: np.ndarray, high: np.ndarray, low: np.ndarray,
    atr_abs: np.ndarray, tp_mult: float, sl_mult: float, horizon: int,
) -> np.ndarray:
    """
    Vectorized TBM. O(horizon × n) numpy ops.
    Returns int8: +1 (TP first), -1 (SL first), 0 (timeout).
    """
    n        = len(close)
    tp_price = close + tp_mult * atr_abs
    sl_price = close - sl_mult * atr_abs
    tp_first = np.full(n, horizon + 1, dtype=np.int32)
    sl_first = np.full(n, horizon + 1, dtype=np.int32)
    t0 = time.perf_counter()
    for k in range(1, horizon + 1):
        if k % 72 == 0:
            print(f"  TBM {k}/{horizon}  {time.perf_counter()-t0:.0f}s", end="\r", flush=True)
        idx = np.arange(n - k, dtype=np.int32)
        tp_m = (tp_first[idx] > k) & (high[idx + k] >= tp_price[idx])
        tp_first[idx[tp_m]] = k
        sl_m = (sl_first[idx] > k) & (low[idx + k] <= sl_price[idx])
        sl_first[idx[sl_m]] = k
    print(f"  TBM done in {time.perf_counter()-t0:.1f}s")
    tp_hit = tp_first <= horizon
    sl_hit = sl_first <= horizon
    labels = np.zeros(n, dtype=np.int8)
    labels[tp_hit & (~sl_hit | (tp_first <= sl_first))] = 1
    labels[sl_hit & (~tp_hit | (sl_first < tp_first))] = -1
    return labels


if LABELS_CACHE.exists():
    print("Loading TBM labels from cache ...")
    labels_full = pd.read_parquet(LABELS_CACHE)["label"].reindex(df.index)
    assert labels_full.notna().all(), "Cache index mismatch — delete cache and rerun"
else:
    print(f"Computing TBM labels on {len(df):,} bars (may take 2-5 min) ...")
    atr_arr    = (df["close"] * df["volat_atr_72_pct"]).values.astype(np.float64)
    raw_labels = compute_tbm_labels(
        df["close"].values.astype(np.float64),
        df["high"].values.astype(np.float64),
        df["low"].values.astype(np.float64),
        atr_arr, TBM_TP_MULT, TBM_SL_MULT, TBM_HORIZON,
    )
    labels_full = pd.Series(raw_labels, index=df.index, name="label", dtype="int8")
    labels_full.to_frame().to_parquet(LABELS_CACHE)
    print(f"  Saved to {LABELS_CACHE}")

df["label"]        = labels_full
df["label_binary"] = (df["label"] == 1).astype(np.int8)
df_train["label"]        = labels_full.reindex(df_train.index)
df_train["label_binary"] = (df_train["label"] == 1).astype(np.int8)
df_val["label"]          = labels_full.reindex(df_val.index)
df_val["label_binary"]   = (df_val["label"] == 1).astype(np.int8)

## Phase B — Label Distribution

In [ ]:
def label_summary(df_split, split_name):
    counts  = df_split["label"].value_counts().sort_index()
    total   = len(df_split)
    pos     = counts.get(1, 0)
    neg     = counts.get(-1, 0)
    timeout = counts.get(0, 0)
    print(f"  {split_name:<14}  "
          f"TP(+1): {pos:>6,} ({pos/total:.1%})  "
          f"SL(-1): {neg:>6,} ({neg/total:.1%})  "
          f"Timeout: {timeout:>6,} ({timeout/total:.1%})  "
          f"Pos rate: {pos/total:.3f}")

print("Label Distribution by Split")
print("-" * 85)
label_summary(df_train, "Train")
label_summary(df_val,   "Validation")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, df_split) in zip(axes, [("Train", df_train), ("Validation 2024", df_val)]):
    counts = df_split["label"].value_counts().sort_index()
    counts.index = ["SL (−1)", "Timeout (0)", "TP (+1)"]
    bars = ax.bar(counts.index, counts.values,
                  color=["#e74c3c", "#95a5a6", "#2ecc71"], alpha=0.85, edgecolor="white")
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(df_split)*0.002,
                f"{val/len(df_split):.1%}", ha="center", fontsize=10)
    ax.set_title(f"TBM Labels — {name}", fontweight="bold")
    ax.set_ylabel("Bar count")
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
plt.tight_layout()
plt.show()

## Phase C — Constrained LightGBM Grid Search

32 configurations × 4 models = **128 LightGBM fits**. Best config per model
selected by validation AUC.

In [ ]:
def train_lgbm_single(X_tr, y_tr, X_vl, y_vl, params):
    model = lgb.LGBMClassifier(**BASE_LGB_PARAMS, **params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=30, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )
    probs = model.predict_proba(X_vl)[:, 1]
    auc   = roc_auc_score(y_vl, probs)
    return model, auc, probs


def run_ablation_grid_search(df_tr, df_vl, ablation_models, param_grid):
    results  = {}
    total    = len(ablation_models) * len(param_grid)
    done     = 0
    t_global = time.perf_counter()
    y_tr = df_tr["label_binary"].values
    y_vl = df_vl["label_binary"].values

    for m_idx, (model_name, feature_list) in enumerate(ablation_models.items(), 1):
        feats = [f for f in feature_list if f in df_tr.columns]
        X_tr  = df_tr[feats].values.astype(np.float32)
        X_vl  = df_vl[feats].values.astype(np.float32)
        best_auc, best_params, best_probs, best_model = -1.0, None, None, None

        print(f"\n[{m_idx}/{len(ablation_models)}] {model_name}  ({len(feats)} features)")

        for c_idx, params in enumerate(param_grid, 1):
            model, auc, probs = train_lgbm_single(X_tr, y_tr, X_vl, y_vl, params)
            done += 1
            elapsed = time.perf_counter() - t_global
            eta     = elapsed / done * (total - done) if done > 0 else 0
            flag    = " ★" if auc > best_auc else ""
            if auc > best_auc:
                best_auc, best_params, best_probs, best_model = auc, params, probs, model
            print(
                f"  cfg {c_idx:>2}/{len(param_grid)}  AUC={auc:.5f}  "
                f"leaves={params['num_leaves']:>2}  depth={params['max_depth']}  "
                f"lr={params['learning_rate']:.3f}  [{done}/{total}  ETA {eta:.0f}s]{flag}",
                flush=True,
            )

        results[model_name] = dict(
            best_auc=best_auc, best_params=best_params,
            best_probs=best_probs, best_model=best_model, n_features=len(feats),
        )
        print(f"  → Best AUC: {best_auc:.5f}")

    print(f"\nGrid search complete: {time.perf_counter()-t_global:.1f}s total")
    return results


ablation_results = run_ablation_grid_search(df_train, df_val, ABLATION_MODELS, PARAM_GRID)

## Phase C — Validation AUC Summary

In [ ]:
auc_rows = []
for model_name, res in ablation_results.items():
    probs     = res["best_probs"]
    n_sig     = (probs > PROB_THRESHOLD).sum()
    n_sig_alt = (probs > PROB_THRESHOLD_ALT).sum()
    auc_rows.append({
        "Model":                              model_name,
        "n_features":                         res["n_features"],
        "Val AUC":                            res["best_auc"],
        f"Signals p>{PROB_THRESHOLD}": n_sig,
        f"Signals/day p>{PROB_THRESHOLD}": n_sig / n_val_days,
    })

auc_df = pd.DataFrame(auc_rows).set_index("Model")
print("Validation AUC & Signal Frequency")
display(auc_df)

## Phase D — Zero-Fee Vectorized Backtest

In [ ]:
def backtest_0fee(df_bt, probs, threshold, tp_mult, sl_mult, horizon):
    close_arr  = df_bt["close"].values.astype(np.float64)
    atr_pct    = df_bt["volat_atr_72_pct"].values.astype(np.float64)
    labels_arr = df_bt["label"].values
    index_arr  = df_bt.index
    n          = len(df_bt)
    eff_thresh = threshold if (probs > threshold).sum() > 0 else PROB_THRESHOLD_ALT
    sig_idx    = np.where(probs > eff_thresh)[0]
    trades        = []
    last_exit_bar = -1
    for i in sig_idx:
        if i <= last_exit_bar:
            continue
        label = labels_arr[i]
        atr_i = atr_pct[i]
        if label == 1:
            pnl_pct, outcome, last_exit_bar = tp_mult * atr_i, "TP", i + 1
        elif label == -1:
            pnl_pct, outcome, last_exit_bar = -sl_mult * atr_i, "SL", i + 1
        else:
            exit_idx = min(i + horizon, n - 1)
            pnl_pct  = (close_arr[exit_idx] - close_arr[i]) / close_arr[i]
            outcome, last_exit_bar = "Timeout", i + horizon
        trades.append({"entry_time": index_arr[i], "label": label,
                       "outcome": outcome, "pnl_pct": pnl_pct,
                       "prob": probs[i], "threshold": eff_thresh})
    return pd.DataFrame(trades).set_index("entry_time") if trades else pd.DataFrame()


def compute_metrics(trades_df, n_days):
    if trades_df.empty:
        return {k: np.nan for k in ["n_trades", "trades_per_day", "win_rate_pct",
                "profit_factor", "ev_per_trade_pct", "total_return_pct"]}
    non_to = trades_df[trades_df["outcome"] != "Timeout"]
    wins   = non_to[non_to["pnl_pct"] > 0]
    losses = non_to[non_to["pnl_pct"] < 0]
    gw     = wins["pnl_pct"].sum()
    gl     = abs(losses["pnl_pct"].sum())
    return {
        "n_trades":         len(trades_df),
        "trades_per_day":   len(trades_df) / n_days,
        "win_rate_pct":     len(wins) / len(non_to) * 100 if len(non_to) > 0 else np.nan,
        "profit_factor":    gw / gl if gl > 0 else np.inf,
        "ev_per_trade_pct": trades_df["pnl_pct"].mean() * 100,
        "total_return_pct": trades_df["pnl_pct"].sum() * 100,
    }


backtest_results = {}
all_trades       = {}

print(f"Running zero-fee backtests on validation set ({n_val_days} days) ...\n")
print(f"{'Model':<25}  {'AUC':>7}  {'Trades':>7}  {'Trd/day':>8}  "
      f"{'WinRate':>8}  {'PF':>6}  {'EV/trd':>8}  {'TotalRet':>9}")
print("-" * 90)

for model_name, res in ablation_results.items():
    trades_df = backtest_0fee(df_val, res["best_probs"], PROB_THRESHOLD,
                              TBM_TP_MULT, TBM_SL_MULT, TBM_HORIZON)
    metrics   = compute_metrics(trades_df, n_val_days)
    metrics["val_auc"] = res["best_auc"]
    backtest_results[model_name] = metrics
    all_trades[model_name]       = trades_df
    print(
        f"{model_name:<25}  {metrics['val_auc']:>7.5f}  "
        f"{metrics['n_trades']:>7.0f}  {metrics['trades_per_day']:>8.2f}  "
        f"{metrics['win_rate_pct']:>7.1f}%  {metrics['profit_factor']:>6.3f}  "
        f"{metrics['ev_per_trade_pct']:>7.3f}%  {metrics['total_return_pct']:>8.2f}%"
    )

## Phase D — Comparative Results Table + Bar Charts

In [ ]:
rows = []
for model_name, metrics in backtest_results.items():
    fee_survivor = (
        not np.isnan(metrics["ev_per_trade_pct"])
        and (metrics["ev_per_trade_pct"] / 100) > EV_SURVIVAL_MIN
    )
    rows.append({
        "Model":          model_name,
        "n_Features":     ablation_results[model_name]["n_features"],
        "Val AUC":        metrics["val_auc"],
        "Trade Count":    metrics["n_trades"],
        "Trades/Day":     metrics["trades_per_day"],
        "Win Rate %":     metrics["win_rate_pct"],
        "Profit Factor":  metrics["profit_factor"],
        "EV/Trade %":     metrics["ev_per_trade_pct"],
        "Total Return %": metrics["total_return_pct"],
        f"Fee Survivor (>{EV_SURVIVAL_MIN:.2%})": "✓ PASS" if fee_survivor else "✗ FAIL",
    })

results_df = pd.DataFrame(rows).set_index("Model")
print("=" * 60)
print("ABLATION STUDY — COMPARATIVE RESULTS (ZERO FEE, VAL 2024)")
print("=" * 60)
display(results_df)

# Bar chart
colors = ["#3498db", "#2ecc71", "#e67e22", "#9b59b6"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("LightGBM Feature Ablation — Zero-Fee Backtest (Validation 2024)",
             fontsize=14, fontweight="bold")

model_labels = results_df.index.tolist()
xpos = np.arange(len(model_labels))

def bar_panel(ax, col, title, ylabel, fmt, hline=None, hline_label=None):
    vals = results_df[col].values.astype(float)
    bars = ax.bar(xpos, vals, color=colors, alpha=0.85, edgecolor="white", width=0.6)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + abs(np.nanmax(vals)-np.nanmin(vals))*0.02,
                    f"{v:{fmt}}", ha="center", va="bottom", fontsize=9)
    if hline is not None:
        ax.axhline(hline, color="red", ls="--", lw=1.2, label=hline_label)
        ax.legend(fontsize=8)
    ax.set_xticks(xpos)
    ax.set_xticklabels([m.replace("_", "\n") for m in model_labels], fontsize=8)
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(axis="y", alpha=0.3)

bar_panel(axes[0,0], "Val AUC",       "Validation AUC",        "AUC",       ".5f")
bar_panel(axes[0,1], "Trade Count",   "Trade Count",           "# trades",  ".0f")
bar_panel(axes[0,2], "Trades/Day",    "Trades per Day",        "trd/day",   ".2f",
          hline=1.5, hline_label="LF mandate")
bar_panel(axes[1,0], "Win Rate %",    "Win Rate %",            "%",         ".1f",
          hline=37.5, hline_label="Break-even")
bar_panel(axes[1,1], "Profit Factor", "Profit Factor",         "PF",        ".3f",
          hline=1.0, hline_label="Break-even")
bar_panel(axes[1,2], "EV/Trade %",    "EV per Trade %",        "% return",  ".3f",
          hline=EV_SURVIVAL_MIN*100, hline_label=f"Fee gate")

plt.tight_layout()
plt.savefig(CACHE_DIR / "lgbm_v0_ablation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Phase D — EV & Fee Survivability Check

In [ ]:
print("=" * 70)
print("FEE SURVIVABILITY ANALYSIS")
print("=" * 70)
print(f"Round-trip fee: {FEE_ROUNDTRIP:.2%}  |  Gate: EV > {EV_SURVIVAL_MIN:.2%}\n")

winners = []
for model_name, metrics in backtest_results.items():
    ev  = metrics["ev_per_trade_pct"]
    fee_mult = (ev / 100) / FEE_ROUNDTRIP if not np.isnan(ev) else np.nan
    passes   = not np.isnan(ev) and (ev / 100) > EV_SURVIVAL_MIN
    if passes:
        winners.append(model_name)
    print(f"  {model_name}")
    print(f"    AUC={metrics['val_auc']:.5f}  Trades={metrics['n_trades']:.0f} ({metrics['trades_per_day']:.2f}/day)")
    print(f"    WR={metrics['win_rate_pct']:.1f}%  PF={metrics['profit_factor']:.3f}  EV={ev:.4f}%  [{fee_mult:.1f}× fee]")
    print(f"    {'✓ PASS' if passes else '✗ FAIL'}\n")

print("=" * 70)
if winners:
    evs  = {w: backtest_results[w]["ev_per_trade_pct"] for w in winners}
    best = max(evs, key=evs.get)
    print(f"PASSED: {winners}")
    print(f"Top model by EV: {best} ({evs[best]:.4f}%)")
    print(f"Best params: {ablation_results[best]['best_params']}")
    print("\nNote: v0 uses original (pre-patch) features. See v1 for leak-corrected results.")
else:
    print("NO MODELS PASSED the fee survivability gate.")
print("=" * 70)